# BubbleFence: ZOD Driving Demo

BubbleFence constructs **bounded semantic regions** ("bubbles") in a foundation-model
embedding space to split streaming video data into train / val / test sets that are
semantically coherent and persist across ingestion rounds.

This notebook runs the full pipeline on dashcam sequences from the
[Zenseact Open Dataset (ZOD)](https://zod.zenseact.com/) Drives subset and visualizes the results.

**Prerequisites** -- download a few ZOD drives:
```bash
pip install zod[cli]
zod download -y --url="<dropbox-url>" --output-dir=zod_data \
    --subset=drives --version=full --no-lidar --no-radar --no-vehicle-data
```
No annotations CSV needed -- BubbleFence auto-scans folders for images.

In [1]:
# On CPU, use --extra-index-url https://download.pytorch.org/whl/cpu instead
!pip install -r bubblefence/requirements.txt --extra-index-url https://download.pytorch.org/whl/rocm7.1 -q

In [1]:
import torch
print(torch.__version__)

2.10.0+rocm7.1


In [ ]:
import sys, os, shutil
from pathlib import Path
import pandas as pd
import yaml
from IPython.display import Image as IPImage, display

sys.path.insert(0, str(Path(".").resolve()))
from bubblefence import load_config, run_batch

print("Imports OK")

## How BubbleFence works -- and where to find each piece

The full pipeline is orchestrated by `run_folder()`, which embeds images,
deduplicates, places anchors, assigns splits, and persists state -- all in one
call.  Everything is controlled by a single YAML config.  The table below maps
each algorithmic concept to the **config key** you would tweak and the
**underlying class** you can import directly if you want to experiment at a
lower level.

| Concept | Config section | Key params | Class |
|---|---|---|---|
| **Embedding** | `foundation_models` | `primary_model` | `FoundationModelProcessor` |
| **Deduplication** | `deduplication` | `similarity_threshold` | handled inside `BubbleFencePipeline` |
| **Density-adaptive warping** | `density_transformation` | `enabled`, `method` | `DensityAnalyzer` |
| **QMC anchor placement** | `anchor_placement` | `method`, `qmc_sequence`, `snap_strategy` | `AnchorPlacer` |
| **Adaptive bubble radii** | `hypersphere` | `radius_computation`, `adaptive_method`, `base_radius_percentile` | `AnchorPlacer` |
| **Nested val/test shells** | `nested_shells` | `enabled`, `validation_ratio` | `AnchorPlacer` |
| **Split targets** | `dataset_splits` | `train_ratio`, `eval_ratio`, `min_eval_per_batch` | `BubbleFencePipeline` |
| **Streaming persistence** | `streaming` | `persistent_anchors`, `anchor_persistence_path` | `BubbleFencePipeline` |
| **Trajectory / embeddings** | -- | -- | `EmbeddingTrajectory` |

Everything is packaged into high-level calls, but the modular design means you
can `from bubblefence import AnchorPlacer` (or any component) and use it
standalone.  We encourage you to dig into the source.

## Configuration

Load the default config and inspect the key parameters.  The YAML file
(`config/bubblefence_config.yaml`) is the single source of truth -- open it
to see every available knob.

In [ ]:
OUTPUT_DIR = "zod_notebook_exp"

# Clean previous run
if Path(OUTPUT_DIR).exists():
    shutil.rmtree(OUTPUT_DIR)

# Print key settings from the BubbleFence config
config = load_config("config/bubblefence_config.yaml")
print(f"Encoder:            {config.foundation_models.primary_model}")
print(f"Device:             {config.embedding.device}")
print(f"Dedup threshold:    {config.deduplication.similarity_threshold}")
print(f"  (0.9999 = near-identical frames only -- keeps trajectory intact for visualization)")
print(f"Anchor placement:   {config.anchor_placement.method} ({config.anchor_placement.qmc_sequence})")
print(f"Snap strategy:      {config.anchor_placement.snap_strategy}")
print(f"Radius:             {config.hypersphere.radius_computation} ({config.hypersphere.adaptive_method})")
print(f"Nested shells:      {config.nested_shells.enabled} (val ratio {config.nested_shells.validation_ratio})")
print(f"Target split:       {config.dataset_splits.train_ratio} train / {config.dataset_splits.eval_ratio} eval")

## Run BubbleFence

`run_batch` reads the YAML config (`config/zod_round1.yaml`) which specifies:
- **folders** to process
- **output_dir** for results
- **bf_config** path to the BubbleFence algorithm config
- **visualizations** to generate after processing

Under the hood, each folder is processed by `run_folder()` which:
1. Auto-discovers images in the folder (no annotations CSV needed)
2. Embeds with the configured foundation model
3. Deduplicates against all previously seen embeddings (cross-batch) and within the batch
4. Assigns points that fall inside existing anchors (from prior runs)
5. Places new anchors via the closed-loop QMC algorithm if the eval ratio needs topping up
6. Persists anchors and embeddings to disk for the next run

Anchors persist across calls -- the second folder's frames that land inside
bubbles from the first folder get assigned automatically.

In [ ]:
run_batch("config/zod_round1.yaml")

## Visualize

`run_batch` already generated the plots specified in the YAML config.
The round 1 config (`config/zod_round1.yaml`) requests:

1. **Trajectory plot** -- t-SNE projection with bubble radius circles and
   auto-selected anchor thumbnails.
2. **Stats** -- dataset statistics printed to stdout (points, splits, anchors).
3. **2-panel summary** -- split assignments and anchor bubble heatmap side by side.

In [ ]:
with open("config/zod_round1.yaml") as f:
    r1_cfg = yaml.safe_load(f)
for vis in r1_cfg["visualizations"]:
    if vis["type"] != "stats":
        display(IPImage(f"{OUTPUT_DIR}/{vis['save']}", width=900))

## Round 2: Incremental dataset growth with persistent anchors

BubbleFence is designed for continual curation -- new data streams in over time
and the split structure grows with it. Anchors and embeddings from Round 1 are
loaded from disk automatically. Incoming frames that fall inside existing
bubbles are assigned instantly; new anchors are only placed if the eval ratio
needs topping up. This is the streaming persistence mechanism at work.

In [ ]:
run_batch("config/zod_round2.yaml")

In [ ]:
with open("config/zod_round2.yaml") as f:
    r2_cfg = yaml.safe_load(f)
for vis in r2_cfg["visualizations"]:
    if vis["type"] != "stats":
        display(IPImage(f"{OUTPUT_DIR}/{vis['save']}", width=900))